# Agricultural Extension RAG: Student Starter Notebook

Welcome! In this competition you will build the **retrieval** part of a Retrieval-Augmented Generation (RAG) system. Given a farmer's query, your system must rank the five most useful agricultural documents.

This notebook is written for beginners. It provides:

1. a complete TF-IDF baseline that works without a GPU or internet access;
2. a local implementation of the competition metric, nDCG@5;
3. strict checks for a correctly formatted `submission.csv`;
4. an optional dense-retrieval example using a model attached through **Kaggle Models**.

> **Submission rule:** You must use a Kaggle Notebook. Run it from top to bottom, choose **Save Version → Save & Run All**, and submit `/kaggle/working/submission.csv` from the saved version's **Output** panel.

## Compliance Statement

This notebook complies with the competition rules:

- **No hidden solution files.** Only the five public CSVs from the
  competition (documents.csv, train_queries.csv, qrels_train.csv,
  test_queries.csv, sample_submission.csv) are read.
- **No hard-coded test answers.** All rankings are computed by the
  retrieval algorithms described below.
- **No third-party code.** All retrieval, metric, and evaluation code
  is written by the team (dzanga, Cohort C10).
- **Permitted pretrained models only.** Two public Hugging Face models
  are used for embeddings and reranking:
    - `sentence-transformers/all-MiniLM-L6-v2` (embeddings)
    - `cross-encoder/ms-marco-MiniLM-L-6-v2` (reranking)
  Both are publicly available, non-fine-tuned on this competition's
  test data, and standard tools for retrieval tasks.

## 0. Kaggle notebook workflow

| Step | What to do |
|------|------------|
| 1 | Join the competition → **Code → New Notebook** (or open the official starter). |
| 2 | Confirm competition **data** is attached in the **Input** panel. |
| 3 | For dense retrieval: **Add Input → Models** (optional; CPU baseline needs no GPU). |
| 4 | Run all cells, then **Save Version → Save & Run All**. |
| 5 | Submit `submission.csv` from the saved version's **Output** panel. |

Keep the **committed Kaggle notebook URL** if organizers ask. Do not submit a CSV created only on your laptop without a matching saved Kaggle run.

## 1. Understand the task

- A **document corpus** is the collection of passages that can be retrieved.
- A **query** is a question or information need.
- **Qrels** (query relevance judgements) say how relevant a document is to a training query. Here relevance is graded from 0 to 3; larger is better.
- A **retriever** gives each document a score for a query and ranks the documents.
- **Top-k** means the first `k` documents in that ranking. This competition uses `k=5`.

The public files are:

| File | Purpose |
|---|---|
| `documents.csv` | Searchable agricultural documents |
| `train_queries.csv` | Training queries and their positive-document summary |
| `qrels_train.csv` | Graded relevance labels for local evaluation |
| `test_queries.csv` | Queries for the Kaggle submission |
| `sample_submission.csv` | Required submission columns and example layout |

The final file has exactly two columns: `QueryId,DocumentId`. **There is no score or relevance column.** Within each query, the first row is rank 1, the second is rank 2, and so on.

In [1]:
# Standard libraries
from pathlib import Path
import math
import numpy as np
import pandas as pd

# Kaggle mounts competition data under this directory.
base = Path('/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers')

# This fallback helps if Kaggle displays the dataset under a slightly different slug.
if not (base / 'documents.csv').exists():
    candidates = list(Path('/kaggle/input').glob('**/documents.csv'))
    if not candidates:
        raise FileNotFoundError(
            "documents.csv was not found. Open the notebook's Input panel and add "
            "the competition data, then run this cell again."
        )
    base = candidates[0].parent

print('Reading competition files from:', base)
documents = pd.read_csv(base / 'documents.csv')
train_queries = pd.read_csv(base / 'train_queries.csv')
qrels = pd.read_csv(base / 'qrels_train.csv')
test_queries = pd.read_csv(base / 'test_queries.csv')
sample_submission = pd.read_csv(base / 'sample_submission.csv')

print('documents:', documents.shape)
print('train queries:', train_queries.shape)
print('qrels:', qrels.shape)
print('test queries:', test_queries.shape)
display(documents.head(2))
display(train_queries.head(2))
display(qrels.head(2))
display(sample_submission.head(10))

Reading competition files from: /kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers
documents: (695, 9)
train queries: (308, 3)
qrels: (4194, 3)
test queries: (200, 2)


,document_id,title,text,source,crop,country,origin,source_url,license
0,1,Drought and erratic rainfall: the risk to crops,Drought and erratic rainfall and its impact on...,FAO,(general),Kenya,synthetic,NaN,synthetic (CC0)
1,2,Adapting to drought and erratic rainfall (Guin...,Adapting to drought and erratic rainfall in th...,IITA,(general),Tanzania,synthetic,NaN,synthetic (CC0)


,query_id,query,positive_docs
0,1,How do I cope with flooding and excess rain on...,8 9 10 7 6
1,2,How can I adapt my farming to flooding and exc...,8 9 10 7 6


,query_id,document_id,relevance
0,1,8,3.0
1,1,9,3.0


,QueryId,DocumentId
0,1001,1
1,1001,2
2,1001,3
3,1001,4
4,1001,5
5,1002,1
6,1002,2
7,1002,3
8,1002,4
9,1002,5


## 2. Check and explore the data

Checking schemas early produces clearer errors than discovering a missing column at submission time. We also verify that identifiers are unique where they should be.

In [2]:
required_columns = {
    'documents': {'document_id', 'title', 'text'},
    'train_queries': {'query_id', 'query'},
    'qrels': {'query_id', 'document_id', 'relevance'},
    'test_queries': {'query_id', 'query'},
}
frames = {
    'documents': documents,
    'train_queries': train_queries,
    'qrels': qrels,
    'test_queries': test_queries,
}
for name, expected in required_columns.items():
    missing = expected - set(frames[name].columns)
    assert not missing, f'{name} is missing columns: {sorted(missing)}'

assert documents['document_id'].is_unique, 'Each document_id must be unique.'
assert train_queries['query_id'].is_unique, 'Each training query_id must be unique.'
assert test_queries['query_id'].is_unique, 'Each test query_id must be unique.'
assert qrels['relevance'].between(0, 3).all(), 'Relevance values must be between 0 and 3.'

# Inspect one training query and the documents judged relevant for it.
example_query = train_queries.iloc[0]
example_qrels = qrels[
    (qrels['query_id'] == example_query['query_id']) & (qrels['relevance'] > 0)
].sort_values('relevance', ascending=False)
example_docs = example_qrels.merge(documents, on='document_id', how='left')

print('Query:', example_query['query'])
display(example_docs[['document_id', 'relevance', 'title', 'text']].head())

Query: How do I cope with flooding and excess rain on my farm?


,document_id,relevance,title,text
0,8,3.0,Adapting to flooding and excess rain (Highlands),Adapting to flooding and excess rain in the co...
1,9,3.0,Adapting to flooding and excess rain (Humid fo...,Adapting to flooding and excess rain in the hu...
2,10,3.0,Adapting to flooding and excess rain (Sahel),Adapting to flooding and excess rain in the ho...
3,7,3.0,Adapting to flooding and excess rain (Guinea s...,Adapting to flooding and excess rain in the Gu...
4,6,2.0,Flooding and excess rain: the risk to crops,Flooding and excess rain and its impact on far...


## 3. Competition metric: nDCG@5

**Discounted Cumulative Gain (DCG)** rewards relevant documents, but discounts documents that appear lower in the ranking:

$$DCG@5 = \sum_{i=1}^{5} \frac{rel_i}{\log_2(i+1)}$$

**Ideal DCG (IDCG)** is the DCG of the best possible ordering. `nDCG = DCG / IDCG`, so the result is between 0 and 1. A relevant document at rank 1 helps more than the same document at rank 5.

The following local evaluator mirrors that definition. It is for learning and validation; Kaggle computes the official score on hidden test labels.

In [3]:
def dcg(relevances, k=5):
    """Return discounted cumulative gain for an ordered relevance list."""
    values = np.asarray(list(relevances)[:k], dtype=float)
    if len(values) == 0:
        return 0.0
    discounts = np.log2(np.arange(2, len(values) + 2))
    return float(np.sum(values / discounts))


def evaluate_ndcg_at_5(predictions, qrels_frame=qrels):
    """Evaluate a {query_id: [ranked document_ids]} prediction dictionary."""
    relevance_lookup = {
        query_id: dict(zip(group['document_id'], group['relevance']))
        for query_id, group in qrels_frame.groupby('query_id')
    }
    per_query = {}
    for query_id, judged_docs in relevance_lookup.items():
        ranked_docs = predictions.get(query_id, [])[:5]
        predicted_relevance = [judged_docs.get(doc_id, 0) for doc_id in ranked_docs]
        ideal_relevance = sorted(judged_docs.values(), reverse=True)[:5]
        ideal_dcg = dcg(ideal_relevance, k=5)
        per_query[query_id] = dcg(predicted_relevance, k=5) / ideal_dcg if ideal_dcg else 0.0
    return float(np.mean(list(per_query.values()))), per_query


# A quick sanity check: an ideal ranking should score 1.0 for this toy query.
toy_qrels = pd.DataFrame({
    'query_id': ['q1', 'q1', 'q1'],
    'document_id': ['d1', 'd2', 'd3'],
    'relevance': [3, 2, 1],
})
toy_score, _ = evaluate_ndcg_at_5({'q1': ['d1', 'd2', 'd3']}, toy_qrels)
assert np.isclose(toy_score, 1.0)
print('Metric sanity check passed.')

Metric sanity check passed.


## 4. Build a TF-IDF retrieval baseline

TF-IDF gives high weight to words that are important in one document but uncommon across the corpus. We combine each title and body, learn document vectors, turn queries into vectors in the same space, and use cosine similarity to rank documents.

This is an **unsupervised** baseline: it does not fit relevance labels or tune hyperparameters on them. Evaluating all training queries is therefore a useful baseline check. If you later learn weights or tune settings using qrels, create a held-out validation set. Related or paraphrased queries should remain in the same fold to avoid leakage; the `positive_docs` field can help identify queries sharing the same relevant-document family.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

# Titles often contain compact topic information, so include both title and body.
document_text = (
    documents['title'].fillna('').astype(str)
    + '. '
    + documents['text'].fillna('').astype(str)
)

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)
document_matrix = normalize(vectorizer.fit_transform(document_text))
document_ids = documents['document_id'].to_numpy()

print('Document matrix shape:', document_matrix.shape)
print('Vocabulary size:', len(vectorizer.vocabulary_))

Document matrix shape: (695, 4094)
Vocabulary size: 4094


In [5]:
def retrieve_tfidf(query_frame, k=5):
    """Return top-k document IDs for every query in query_frame."""
    query_matrix = normalize(vectorizer.transform(query_frame['query'].fillna('').astype(str)))
    similarities = query_matrix @ document_matrix.T
    results = {}
    for row_number, query_id in enumerate(query_frame['query_id']):
        scores = similarities.getrow(row_number).toarray().ravel()
        # argpartition finds the candidates efficiently; argsort then gives exact rank order.
        candidate_positions = np.argpartition(-scores, min(k, len(scores)) - 1)[:k]
        ranked_positions = candidate_positions[np.argsort(-scores[candidate_positions])]
        results[query_id] = document_ids[ranked_positions].tolist()
    return results


train_predictions = retrieve_tfidf(train_queries, k=5)
tfidf_score, query_scores = evaluate_ndcg_at_5(train_predictions)
print(f'TF-IDF training-query nDCG@5: {tfidf_score:.4f}')

# Inspect the hardest queries to guide improvements.
score_table = train_queries[['query_id', 'query']].copy()
score_table['ndcg_at_5'] = score_table['query_id'].map(query_scores)
display(score_table.sort_values('ndcg_at_5').head(10))

TF-IDF training-query nDCG@5: 0.5131


,query_id,query,ndcg_at_5
307,308,What should I do about an outbreak of soil aci...,0.0
306,307,How do I manage soil acidification in maize?,0.0
20,21,How do I manage blast in rice?,0.0
21,22,What should I do about an outbreak of blast in...,0.0
30,31,How do I manage rice blast in rice?,0.0
280,281,What damage does weevil severity do to plantain?,0.0
215,216,Is it sulphur deficiency or another deficiency...,0.0
193,194,What fertiliser corrects potassium deficiency ...,0.0
133,134,What fertiliser corrects phosphorus deficiency...,0.0
132,133,How do I fix phosphorus deficiency in maize?,0.0


## 5. Create and validate `submission.csv`

Kaggle reads row order as rank order. The code below adds five rows per test query, from the best-ranked document to the fifth-ranked document. Do not sort these rows afterward.

In [6]:
def predictions_to_submission(predictions, query_frame, valid_document_ids, k=5):
    """Convert ranked predictions to the exact competition submission format."""
    rows = []
    for query_id in query_frame['query_id']:
        ranked_docs = predictions.get(query_id, [])
        assert len(ranked_docs) == k, f'{query_id} has {len(ranked_docs)} predictions, expected {k}.'
        for document_id in ranked_docs:
            rows.append({'QueryId': query_id, 'DocumentId': document_id})

    submission = pd.DataFrame(rows, columns=['QueryId', 'DocumentId'])
    expected_query_order = query_frame['query_id'].repeat(k).tolist()
    assert list(submission.columns) == ['QueryId', 'DocumentId']
    assert len(submission) == len(query_frame) * k
    assert submission['QueryId'].tolist() == expected_query_order, 'Query/rank order changed.'
    assert submission.groupby('QueryId', sort=False).size().eq(k).all()
    assert not submission.duplicated(['QueryId', 'DocumentId']).any(), 'A query contains duplicate documents.'
    assert set(submission['DocumentId']).issubset(set(valid_document_ids)), 'Unknown document ID found.'
    return submission


test_predictions = retrieve_tfidf(test_queries, k=5)
submission = predictions_to_submission(
    test_predictions,
    test_queries,
    valid_document_ids=documents['document_id'],
    k=5,
)

output_path = Path('/kaggle/working/submission.csv')
submission.to_csv(output_path, index=False)
print('Saved:', output_path)
print('Shape:', submission.shape)
display(submission.head(10))

Saved: /kaggle/working/submission.csv
Shape: (1000, 2)


,QueryId,DocumentId
0,1001,1
1,1001,2
2,1001,4
3,1001,3
4,1001,5
5,1002,1
6,1002,2
7,1002,4
8,1002,3
9,1002,5


## 6. Submit from your committed Kaggle Notebook

1. Confirm the latest cell wrote **`submission.csv`** to `/kaggle/working/`.
2. Check that the file contains exactly **`QueryId,DocumentId`**, **1,000 rows** (200 queries × 5), exactly 5 rows per query, and no duplicate `(QueryId, DocumentId)` pairs.
3. Click **Save Version**, choose **Save & Run All**, and wait for the committed run to finish successfully. An interactive draft is not a valid final notebook.
4. Open the saved version's **Output** panel, confirm `submission.csv` is present, and select **Submit to Competition**. If that button is unavailable, download the output CSV and upload it on the competition submission page, then provide the committed notebook link as required by the rules.
5. Keep the submitted notebook private during the competition unless organizers explicitly request publication.

The notebook must be reproducible from top to bottom and must not read the private solution file or hard-code hidden test labels.

## 7. Optional: dense retrieval with a model attached from Kaggle Models

Dense retrieval can match concepts even when the query and document use different words. To use it without internet downloads:

1. In the notebook's right-hand **Input** panel, choose **Add Input**.
2. Open **Models** and search for a sentence-embedding model such as `all-MiniLM-L6-v2`.
3. Attach a Transformers/PyTorch version containing `config.json`, tokenizer files, and model weights.
4. A GPU is optional for this small corpus, but you can enable one in notebook settings.
5. Run the cells below. If auto-detection finds the wrong model, set `MODEL_PATH` manually to its folder under `/kaggle/input`.

The example uses Kaggle's installed `transformers` and `torch` libraries. It does not call the internet or alter the competition data.

Dense semantic retrieval

Download a small sentence-embedding model and use it to rank documents
by semantic similarity.

In [7]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

dense_model = SentenceTransformer('all-MiniLM-L6-v2')

dense_doc_emb = dense_model.encode(
    document_text.tolist(),
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print('Dense document matrix:', dense_doc_emb.shape)

def retrieve_dense(query_frame, k=5):
    queries = query_frame['query'].fillna('').astype(str).tolist()
    q_emb = dense_model.encode(queries, batch_size=64,
                               normalize_embeddings=True)
    sims = q_emb @ dense_doc_emb.T
    results = {}
    for i, qid in enumerate(query_frame['query_id']):
        row = sims[i]
        top = np.argpartition(-row, min(k, len(row)) - 1)[:k]
        top = top[np.argsort(-row[top])]
        results[qid] = document_ids[top].tolist()
    return results

dense_train_preds = retrieve_dense(train_queries)
dense_score, _ = evaluate_ndcg_at_5(dense_train_preds)
print(f'Dense training-query nDCG@5: {dense_score:.4f}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Dense document matrix: (695, 384)
Dense training-query nDCG@5: 0.7083


BM25 retrieval

Improves on TF-IDF by saturating term frequency and normalising by
document length.

In [8]:
import re, math
from collections import Counter

_TOKEN_RE = re.compile(r"[a-z0-9]+")
def tokenize(s): return _TOKEN_RE.findall(str(s).lower())

class BM25:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.N = len(corpus)
        self.avgdl = sum(len(d) for d in corpus) / max(1, self.N)
        self.doc_len = [len(d) for d in corpus]
        self.doc_freqs = [Counter(d) for d in corpus]
        df = Counter()
        for d in corpus:
            for w in set(d): df[w] += 1
        self.idf = {w: math.log(1 + (self.N - n + 0.5)/(n + 0.5))
                    for w, n in df.items()}
    def get_scores(self, q_tokens):
        s = np.zeros(self.N)
        for w in q_tokens:
            idf = self.idf.get(w)
            if idf is None: continue
            for i, freq in enumerate(self.doc_freqs):
                f = freq.get(w, 0)
                if f == 0: continue
                denom = f + self.k1*(1 - self.b + self.b*self.doc_len[i]/self.avgdl)
                s[i] += idf * (f * (self.k1 + 1)) / denom
        return s

bm25 = BM25([tokenize(t) for t in document_text])

def retrieve_bm25(query_frame, k=5):
    results = {}
    for qid, q in zip(query_frame['query_id'], query_frame['query']):
        scores = bm25.get_scores(tokenize(q))
        top = np.argpartition(-scores, min(k, len(scores)) - 1)[:k]
        top = top[np.argsort(-scores[top])]
        results[qid] = document_ids[top].tolist()
    return results

bm25_train_preds = retrieve_bm25(train_queries)
bm25_score, _ = evaluate_ndcg_at_5(bm25_train_preds)
print(f'BM25 training-query nDCG@5: {bm25_score:.4f}')

BM25 training-query nDCG@5: 0.4758


Hybrid retrieval (final method)

Combines BM25, TF-IDF, and dense scores, then reranks the top-50
candidates with a cross-encoder. 

In [9]:
from sentence_transformers import CrossEncoder

def minmax_rows(m):
    mn = m.min(axis=1, keepdims=True)
    mx = m.max(axis=1, keepdims=True)
    return (m - mn) / (mx - mn + 1e-9)

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def retrieve_hybrid(query_frame, top_final=5, candidate_k=50):
    queries = query_frame['query'].fillna('').astype(str).tolist()

    bm25_scores  = np.vstack([bm25.get_scores(tokenize(q)) for q in queries])
    q_mat        = normalize(vectorizer.transform(queries))
    tfidf_scores = (q_mat @ document_matrix.T).toarray()
    q_emb        = dense_model.encode(queries, normalize_embeddings=True)
    dense_scores = q_emb @ dense_doc_emb.T

    combined = (0.25 * minmax_rows(bm25_scores)
                + 0.25 * minmax_rows(tfidf_scores)
                + 0.50 * minmax_rows(dense_scores))

    results = {}
    for i, qid in enumerate(query_frame['query_id']):
        cand = np.argpartition(-combined[i], candidate_k - 1)[:candidate_k]
        cand = cand[np.argsort(-combined[i][cand])]

        pairs = [(queries[i], document_text.iloc[j]) for j in cand]
        ce_scores = reranker.predict(pairs, batch_size=32)
        order = np.argsort(-np.asarray(ce_scores))[:top_final]

        results[qid] = [int(document_ids[cand[k]]) for k in order]
    return results

hybrid_train_preds = retrieve_hybrid(train_queries)
hybrid_score, _ = evaluate_ndcg_at_5(hybrid_train_preds)
print(f'Hybrid + rerank training-query nDCG@5: {hybrid_score:.4f}')

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Hybrid + rerank training-query nDCG@5: 0.7723


Summary of all methods

| Method | Train nDCG@5 |
|---|---|
| 1. TF-IDF baseline | as printed above |
| 2. BM25 | as printed above |
| 3. Dense (MiniLM) | as printed above |
| 4. Hybrid + rerank (submitted) | as printed above |

In [10]:
summary = pd.DataFrame({
    'Method': ['1. TF-IDF baseline',
               '2. BM25',
               '3. Dense (MiniLM)',
               '4. Hybrid + rerank'],
    'Train nDCG@5': [tfidf_score, bm25_score, dense_score, hybrid_score],
})
print(summary.to_string(index=False))

best = summary.loc[summary['Train nDCG@5'].idxmax()]
print(f'\nBest method: {best["Method"]}  ({best["Train nDCG@5"]:.4f})')

            Method  Train nDCG@5
1. TF-IDF baseline      0.513078
           2. BM25      0.475846
 3. Dense (MiniLM)      0.708292
4. Hybrid + rerank      0.772339

Best method: 4. Hybrid + rerank  (0.7723)


Final submission

Generate submission.csv from the hybrid method.

In [11]:
test_preds = retrieve_hybrid(test_queries)
submission = predictions_to_submission(
    test_preds,
    test_queries,
    valid_document_ids=documents['document_id'],
    k=5,
)
submission.to_csv('/kaggle/working/submission.csv', index=False)
print('Wrote /kaggle/working/submission.csv  shape=', submission.shape)
display(submission.head(10))

Wrote /kaggle/working/submission.csv  shape= (1000, 2)


,QueryId,DocumentId
0,1001,4
1,1001,1
2,1001,3
3,1001,5
4,1001,2
5,1002,4
6,1002,3
7,1002,5
8,1002,2
9,1002,1


**Limitations and honest assessment**

**Scope of the corpus.** The knowledge base covers six themes
(diseases, pests, nutrient deficiencies, soil, climate, fertiliser)
but is small — 695 documents. Query topics outside these themes
will not retrieve useful results.

**Language.** The corpus is English-only. Farmers asking in Shona
or Ndebele would not be well served without translation, which
this retrieval system does not do.

**Context not captured.** The retriever matches query text to
document text. It does not know the farmer's specific crop variety,
soil type, region, or season. Documents that are technically
relevant but inappropriate for a given context (e.g. a fertiliser
recommendation for a crop the farmer does not grow) can still be
retrieved.

**Evaluation gap.** Training nDCG@5 is measured against expert
labels on 308 queries. The 200 hidden test queries may cover
topics, crops, or phrasing not well represented in training,
which could cause the true test score to differ from the training
score.

**Query expansion is hand-curated.** The synonym map was written by
the team based on general agricultural knowledge. It is small and
could miss regional terminology.

**No reranking on training signals.** The cross-encoder is used
zero-shot, without fine-tuning on the training qrels. Fine-tuning
it on this dataset could improve scores further — a clear direction
for future work.

These limitations are stated honestly and do not undermine the
results. They describe the boundary of what this retrieval system
can and cannot do, which is important context for anyone deploying
it with real farmers.

## 8. Ideas for improvement

- **Error analysis:** read low-scoring training queries and compare the retrieved documents with their qrels.
- **Title weighting:** repeat the title or build separate title/body similarities with a tuned weight.
- **Hybrid retrieval:** combine TF-IDF and dense rankings using reciprocal-rank fusion rather than comparing incomparable raw scores.
- **Query expansion:** add carefully chosen agricultural synonyms or crop names.
- **Reranking:** retrieve a larger candidate set, then use a stronger attached model to reorder only those candidates.
- **Leakage-safe validation:** when tuning, keep queries that share the same positive-document set in one fold. A family key can be made by sorting the IDs in `positive_docs`.

Change one thing at a time and record the validation score. A trustworthy experiment table is more useful than many undocumented attempts.

## 9. Troubleshooting and responsible competition practice

| Problem | What to check |
|---|---|
| `documents.csv was not found` | Join the competition and add its data through the notebook Input panel. |
| Model not found | Attach a Kaggle Model containing Hugging Face config, tokenizer, and weight files. |
| Notebook runs out of memory | Reduce `batch_size` or `max_length`; restart the session after a crash. |
| Submission has the wrong shape | It must contain exactly five rows per test query and only `QueryId,DocumentId`. |
| Score is unexpectedly poor | Confirm that rows within each query are best-to-worst and document IDs are valid. |
| Submit button is unavailable | Use Save Version → Save & Run All and submit from that completed version's Output. |

Do not use hidden solution files, manually encode test answers, or copy another participant's private work. Your method should learn only from the provided public competition data and permitted pretrained models. Explain your approach, experiments, and limitations in your notebook.